# A shrinking bracket around a pole
Compare a continuous zero crossing with a discontinuous sign reversal at exactly the same location. Keep a separate record of spatial width and measured residual on every iteration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

location = np.sqrt(2)/4
rounds = 20
assert 0 < location < 1 and 5 <= rounds <= 35
def bisect_record(f, tie_residual):
    lo,hi=0.,1.
    data=[]
    for j in range(rounds):
        mid=(lo+hi)/2
        if mid==location:
            data.append((mid,hi-lo,tie_residual))
            break
        value=f(mid)
        data.append((mid,hi-lo,abs(value)))
        if np.signbit(value)==np.signbit(f(lo)):lo=mid
        else:hi=mid
    return np.array(data)
zero=bisect_record(lambda x:x-location,0.)
pole=bisect_record(lambda x:1/(x-location),np.inf)
regular=np.isfinite(pole[:,2])
assert np.allclose(zero[:,:2],pole[:,:2])
assert np.allclose(zero[regular,2]*pole[regular,2],1)
if not regular[-1]: print("Exact midpoint hit: the linear signal vanishes and the reciprocal is undefined.")
print("Final width",zero[-1,1],"zero residual",zero[-1,2],"pole magnitude",pole[-1,2])

## Change the observable as well as the tolerance
Both candidates go through identical midpoints, although only one residual can approach zero. Take two new distances from the candidate and compare how the output scales.

In [ ]:
fig,(shape,error)=plt.subplots(1,2,figsize=(10,4),layout="constrained")
margin = min(.01,location/2,(1-location)/2)
for xs in (np.linspace(0,location-margin,180),np.linspace(location+margin,1,240)):
    shape.plot(xs,1/(xs-location),color="#b94626")
shape.plot([0,1],[-location,1-location],color="#145c71",label="continuous crossing")
shape.plot([],[],color="#b94626",label="pole")
shape.set(xlabel="x",ylabel="Signal",ylim=(-10,10),title="Same endpoint signs")
shape.legend()
j=np.arange(1,len(zero)+1)
for data,c,label in [(zero,"#145c71","absolute zero residual"),(pole,"#b94626","absolute pole residual")]:
    error.semilogy(j[regular],data[regular,2],".-",color=c,label=label)
error.semilogy(j,zero[:,1],":",color="black",label="bracket width")
error.set(xlabel="Iteration",ylabel="Magnitude",title="A width test cannot certify continuity")
error.legend(fontsize=8)
for ax in (shape,error):ax.grid(alpha=.2)
plt.show()